# 08 - ML System Monitoring

Makes the deployed system observable. This notebook scores the production
data with the project's model, logs each prediction to CloudWatch Logs,
publishes summary metrics, runs a data drift check (PSI) against the
training baseline, and builds a CloudWatch dashboard and alarms.

This is the operability evidence for the screencast.

**Prerequisites:** Run `04_split.ipynb` first. This notebook depends on:
- splits in `s3://<bucket>/splits/` (written by notebook 04)
- `project_config.json` (shared config used by notebooks 04-07)

**Design note:** Monitoring scores the production split with a locally rebuilt
copy of the model rather than unpickling the registered artifact. This keeps
the notebook independent of the notebook 06 run state and avoids SKLearn
pickle version skew, the same independence principle used in notebook 07.
In production the same logging would wrap the live endpoint or batch transform
output. The model config matches the best model selected in notebook 05.

## 0. Install Dependencies

In [ ]:
import importlib
import subprocess
import sys


def install_if_missing(package, import_name=None):
    name = import_name or package
    if importlib.util.find_spec(name) is None:
        print(f'Installing {package}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', package, '--quiet'], check=True)
        print(f'{package} installed')
    else:
        print(f'{package} already installed, skipping')


for pkg, imp in [('boto3', 'boto3'), ('pandas', 'pandas'),
                 ('scikit-learn', 'sklearn'), ('pyarrow', 'pyarrow')]:
    install_if_missing(pkg, imp)

print('Dependencies ready')

## 1. Setup

Loads the shared config, resolves the region the project resources live in,
and creates the boto3 clients. Region and bucket are read from
`project_config.json` so this notebook follows whatever the rest of the repo
uses. If you see resources created in the wrong account or region, reconcile
`project_config.json` first, because the SageMaker resources from notebooks
06-07 live in a specific region and the monitoring must match it.

In [ ]:
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import boto3
import numpy as np
import pandas as pd

# Same default + override pattern used in notebooks 04-07
default_config = {
    "REGION": "us-east-1",
    "SOURCE_BUCKET": "aai540-group1-yelp-data",
    "FEATURE_COLS": ["review_length", "word_count", "useful", "funny", "cool", "vader_score"],
    "TARGET_COL": "sentiment",
    "RANDOM_STATE": 42,
}

config_path = Path("project_config.json")
if config_path.exists():
    with config_path.open() as f:
        cfg = json.load(f)
    print("Loaded project_config.json")
else:
    cfg = dict(default_config)
    print("project_config.json not found, using defaults")

session = boto3.session.Session()
REGION = cfg.get("REGION") or session.region_name or default_config["REGION"]
SOURCE_BUCKET = cfg.get("SOURCE_BUCKET", default_config["SOURCE_BUCKET"])
FEATURE_COLS = cfg.get("FEATURE_COLS", default_config["FEATURE_COLS"])
TARGET_COL = cfg.get("TARGET_COL", default_config["TARGET_COL"])
RANDOM_STATE = cfg.get("RANDOM_STATE", default_config["RANDOM_STATE"])

# Monitoring resource names, reused across cells and saved back to config
LOG_GROUP = "/yelp-sentiment/predictions"
LOG_STREAM = "batch-" + datetime.now(timezone.utc).strftime("%Y-%m-%d-%H-%M-%S")
METRIC_NAMESPACE = "YelpSentiment/Monitoring"
DASHBOARD_NAME = "yelp-sentiment-monitoring"
CONFIDENCE_ALARM = "yelp-sentiment-low-confidence"
DRIFT_ALARM = "yelp-sentiment-feature-drift"

s3 = boto3.client("s3", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cloudwatch = boto3.client("cloudwatch", region_name=REGION)

print("Region        :", REGION)
print("Source bucket :", SOURCE_BUCKET)
print("Log group     :", LOG_GROUP)
print("Namespace     :", METRIC_NAMESPACE)

## 2. Load Data and Score the Production Stream

Loads the train and production splits, rebuilds the model with the same
configuration as the best model in notebook 05 (Random Forest, balanced class
weights), and scores the production set. The scored output is the prediction
stream the monitoring layer observes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier


def load_split(split_name):
    local = f"/tmp/{split_name}.parquet"
    s3.download_file(SOURCE_BUCKET, f"splits/{split_name}.parquet", local)
    return pd.read_parquet(local)


print(f"Loading splits from s3://{SOURCE_BUCKET}/splits/ ...")
train_data = load_split("train")
prod_data = load_split("production")
print(f"  Train      : {len(train_data):,} rows")
print(f"  Production : {len(prod_data):,} rows")

X_train, y_train = train_data[FEATURE_COLS], train_data[TARGET_COL]
X_prod, y_prod = prod_data[FEATURE_COLS], prod_data[TARGET_COL]

# Rebuild the production model (matches -->> the best model in notebook 05)
model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_train, y_train)

prod_prob = model.predict_proba(X_prod)[:, 1]      # P(positive)
prod_pred = (prod_prob >= 0.5).astype(int)
prod_conf = np.maximum(prod_prob, 1 - prod_prob)   # distance from the 0.5 boundary

print("\nScored production set:")
print(f"  Predicted positive rate : {prod_pred.mean() * 100:.1f}%")
print(f"  Mean confidence         : {prod_conf.mean():.3f}")

## 3. Log Predictions to CloudWatch Logs

Creates the log group and a run-specific stream, then writes a sample of the
prediction stream as structured JSON events. Each event carries the review id,
predicted label, model score, confidence, and the true label. Timestamps are
spread across the last hour so the dashboard shows a time series.

In [ ]:
# Create the log group and stream
try:
    logs.create_log_group(logGroupName=LOG_GROUP)
    print(f"Created log group {LOG_GROUP}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"Log group {LOG_GROUP} already exists")

try:
    logs.create_log_stream(logGroupName=LOG_GROUP, logStreamName=LOG_STREAM)
    print(f"Created log stream {LOG_STREAM}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"Log stream {LOG_STREAM} already exists")

# Build a sample prediction stream to log. A sample keeps the call fast.
SAMPLE_SIZE = 1000
sample = prod_data.head(SAMPLE_SIZE).reset_index(drop=True)
sample_prob = prod_prob[:SAMPLE_SIZE]
sample_pred = prod_pred[:SAMPLE_SIZE]
sample_conf = prod_conf[:SAMPLE_SIZE]

now_ms = int(time.time() * 1000)
start_ms = now_ms - 60 * 60 * 1000
step = max((now_ms - start_ms) // max(len(sample), 1), 1)

events = []
for i, row in sample.iterrows():
    events.append({
        "timestamp": start_ms + i * step,
        "message": json.dumps({
            "review_id": str(row.get("review_id", f"row-{i}")),
            "predicted_label": "positive" if sample_pred[i] == 1 else "negative",
            "score": round(float(sample_prob[i]), 4),
            "confidence": round(float(sample_conf[i]), 4),
            "true_label": "positive" if row[TARGET_COL] == 1 else "negative",
        }),
    })


# put_log_events accepts up to 10,000 events-->>  send in chunks
def put_events(event_batch, token=None):
    kwargs = {"logGroupName": LOG_GROUP, "logStreamName": LOG_STREAM, "logEvents": event_batch}
    if token:
        kwargs["sequenceToken"] = token
    return logs.put_log_events(**kwargs)


token = None
CHUNK = 500
for j in range(0, len(events), CHUNK):
    resp = put_events(events[j:j + CHUNK], token)
    token = resp.get("nextSequenceToken")

print(f"Logged {len(events)} prediction events to {LOG_GROUP}")

## 4. Publish Summary Metrics

Pushes a few rollup metrics to a custom namespace so they can be charted on
the dashboard and watched by alarms.

In [ ]:
metric_time = datetime.now(timezone.utc)


def put_metric(name, value, unit="None"):
    cloudwatch.put_metric_data(
        Namespace=METRIC_NAMESPACE,
        MetricData=[{
            "MetricName": name,
            "Timestamp": metric_time,
            "Value": float(value),
            "Unit": unit,
        }],
    )


put_metric("PredictionVolume", len(prod_pred), "Count")
put_metric("PositivePredictionRate", float(prod_pred.mean()) * 100, "Percent")
put_metric("MeanConfidence", float(prod_conf.mean()))

print("Published metrics to namespace", METRIC_NAMESPACE)
print(f"  PredictionVolume       = {len(prod_pred)}")
print(f"  PositivePredictionRate = {prod_pred.mean() * 100:.1f}%")
print(f"  MeanConfidence         = {prod_conf.mean():.3f}")

## 5. Data Drift Check (PSI)

Population Stability Index compares each feature's distribution in production
against the training baseline. 

Standard reading: PSI below 0.10 is no significant shift, 0.10 to 0.25 is a
moderate shift, above 0.25 is a significant shift that warrants retraining.

In [ ]:
def population_stability_index(baseline, current, bins=10):
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(baseline, quantiles))
    if len(edges) < 3:                # near-constant feature
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    base_pct = np.histogram(baseline, bins=edges)[0] / len(baseline)
    curr_pct = np.histogram(current, bins=edges)[0] / len(current)
    base_pct = np.clip(base_pct, 1e-6, None)
    curr_pct = np.clip(curr_pct, 1e-6, None)
    return float(np.sum((curr_pct - base_pct) * np.log(curr_pct / base_pct)))


def psi_verdict(psi):
    if psi < 0.10:
        return "no significant shift"
    if psi < 0.25:
        return "moderate shift"
    return "significant shift"


print("Feature drift (train baseline vs production):\n")
print(f"{'feature':<16}{'PSI':>8}   verdict")
print("-" * 46)
drift_rows = []
for col in FEATURE_COLS:
    psi = population_stability_index(X_train[col].values, X_prod[col].values)
    drift_rows.append((col, psi))
    print(f"{col:<16}{psi:>8.4f}   {psi_verdict(psi)}")

# Prediction drift: score distribution baseline vs production
pred_psi = population_stability_index(model.predict_proba(X_train)[:, 1], prod_prob)
print("-" * 46)
print(f"{'prediction':<16}{pred_psi:>8.4f}   {psi_verdict(pred_psi)}")

max_psi = max(p for _, p in drift_rows)
put_metric("MaxFeaturePSI", max_psi)
print(f"\nMax feature PSI = {max_psi:.4f} ({psi_verdict(max_psi)})")

## 6. Create the CloudWatch Dashboard

Two metric widgets and a Logs Insights table. Open this dashboard in the
console for the screencast.

In [ ]:
dashboard_body = {
    "widgets": [
        {
            "type": "metric", "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "title": "Prediction volume and positive rate",
                "region": REGION,
                "metrics": [
                    [METRIC_NAMESPACE, "PredictionVolume", {"stat": "Sum"}],
                    [METRIC_NAMESPACE, "PositivePredictionRate", {"stat": "Average", "yAxis": "right"}],
                ],
                "view": "timeSeries", "period": 300,
            },
        },
        {
            "type": "metric", "x": 12, "y": 0, "width": 12, "height": 6,
            "properties": {
                "title": "Mean confidence and max feature PSI",
                "region": REGION,
                "metrics": [
                    [METRIC_NAMESPACE, "MeanConfidence", {"stat": "Average"}],
                    [METRIC_NAMESPACE, "MaxFeaturePSI", {"stat": "Average", "yAxis": "right"}],
                ],
                "view": "timeSeries", "period": 300,
            },
        },
        {
            "type": "log", "x": 0, "y": 6, "width": 24, "height": 6,
            "properties": {
                "title": "Recent prediction log",
                "region": REGION,
                "query": "fields @timestamp, predicted_label, confidence, true_label | sort @timestamp desc | limit 50",
                "logGroupNames": [LOG_GROUP],
                "view": "table",
            },
        },
    ],
}

cloudwatch.put_dashboard(DashboardName=DASHBOARD_NAME, DashboardBody=json.dumps(dashboard_body))
console_url = (
    f"https://{REGION}.console.aws.amazon.com/cloudwatch/home"
    f"?region={REGION}#dashboards:name={DASHBOARD_NAME}"
)
print(f"Created dashboard '{DASHBOARD_NAME}'")
print("View it here:", console_url)

## 7. Create Alarms

One alarm on mean confidence and one on feature drift. With only a single data
point published they may sit in INSUFFICIENT_DATA until the scheduled pipeline
publishes more, which is expected. Missing data is treated as not breaching.

In [ ]:
cloudwatch.put_metric_alarm(
    AlarmName=CONFIDENCE_ALARM,
    AlarmDescription="Mean prediction confidence dropped below 0.65, possible model or data issue.",
    Namespace=METRIC_NAMESPACE,
    MetricName="MeanConfidence",
    Statistic="Average",
    Period=300,
    EvaluationPeriods=1,
    Threshold=0.65,
    ComparisonOperator="LessThanThreshold",
    TreatMissingData="notBreaching",
)
print(f"Created alarm '{CONFIDENCE_ALARM}' (MeanConfidence < 0.65)")

cloudwatch.put_metric_alarm(
    AlarmName=DRIFT_ALARM,
    AlarmDescription="Max feature PSI exceeded 0.25, significant data drift.",
    Namespace=METRIC_NAMESPACE,
    MetricName="MaxFeaturePSI",
    Statistic="Average",
    Period=300,
    EvaluationPeriods=1,
    Threshold=0.25,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
)
print(f"Created alarm '{DRIFT_ALARM}' (MaxFeaturePSI > 0.25)")

## 8. Save Config and Summary

Records the monitoring resource names back into `project_config.json` so the System Design Document and
other notebooks can reference them.

In [ ]:
cfg["MONITORING"] = {
    "LOG_GROUP": LOG_GROUP,
    "METRIC_NAMESPACE": METRIC_NAMESPACE,
    "DASHBOARD_NAME": DASHBOARD_NAME,
    "ALARMS": [CONFIDENCE_ALARM, DRIFT_ALARM],
}
with open("project_config.json", "w") as f:
    json.dump(cfg, f, indent=2)
print("Updated project_config.json with monitoring resources\n")

print("=== Monitoring summary ===")
print(f"Predictions scored : {len(prod_pred):,}")
print(f"Positive rate      : {prod_pred.mean() * 100:.1f}%")
print(f"Mean confidence    : {prod_conf.mean():.3f}")
print(f"Max feature PSI    : {max_psi:.4f} ({psi_verdict(max_psi)})")
print(f"Log group          : {LOG_GROUP}")
print(f"Dashboard          : {DASHBOARD_NAME}")
print("\nFor the screencast: open the CloudWatch dashboard above and the")
print("Logs Insights view to show the live prediction feed.")